In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from scipy import ndimage, stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import pandas as pd

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

C_BLUE, C_ORANGE, C_AQUA, C_GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#9a9a94"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

In [ ]:
MEAN, STD = 0.2860, 0.3530
def normalize(x):                       # x in [0, 1]
    return (x - MEAN) / STD

tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])
train_ds = datasets.FashionMNIST("./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.FashionMNIST("./data", train=False, download=True, transform=tfm)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=512, shuffle=False)

CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
X_test_raw = test_ds.data.float() / 255.0      # (10000, 28, 28) in [0,1]
y_test     = test_ds.targets.clone()

fig, axes = plt.subplots(1, 10, figsize=(13, 1.8))
for k, ax in enumerate(axes):
    ax.imshow(X_test_raw[(y_test == k).nonzero()[0].item()], cmap="gray")
    ax.set_title(CLASSES[k], fontsize=8); ax.axis("off")
plt.show()

In [ ]:
class SmallCNN(nn.Module):
    LAYERS = ["conv1", "conv2", "conv3"]
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)

    def features_upto(self, x, layer):
        a = F.relu(self.conv1(x))
        if layer == "conv1": return a
        a = F.relu(self.conv2(self.pool(a)))
        if layer == "conv2": return a
        return F.relu(self.conv3(self.pool(a)))

    def forward_from(self, a, layer):
        if layer == "conv1":
            a = F.relu(self.conv2(self.pool(a))); layer = "conv2"
        if layer == "conv2":
            a = F.relu(self.conv3(self.pool(a)))
        a = torch.flatten(a, 1)
        return self.fc2(F.relu(self.fc1(a)))

    def forward(self, x):
        return self.forward_from(self.features_upto(x, "conv3"), "conv3")

@torch.no_grad()
def predict(net, X_raw, batch=512):
    """X_raw: (N,28,28) tensor in [0,1]. Returns predicted labels."""
    net.eval()
    out = []
    for i in range(0, len(X_raw), batch):
        x = normalize(X_raw[i:i+batch, None]).to(device)
        out.append(net(x).argmax(1).cpu())
    return torch.cat(out)

def evaluate(net):
    return (predict(net, X_test_raw) == y_test).float().mean().item()

model = SmallCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(3):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch+1}: test accuracy = {evaluate(model):.4f}")
model.eval();

In [ ]:
# Part 1.3 — Baseline behaviour

preds = predict(model, X_test_raw)
cm = confusion_matrix(y_test.numpy(), preds.numpy(), labels=np.arange(10))

# (a) Per-class accuracy
per_class_acc = np.array([
    ((preds[y_test == k] == k).float().mean().item())
    for k in range(10)
])

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CLASSES, per_class_acc)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_xlabel("Fashion-MNIST class")
ax.set_title("Per-class test accuracy")
ax.tick_params(axis="x", rotation=35)
for b, v in zip(bars, per_class_acc):
    ax.text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.3f}",
            ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

print("Overall test accuracy:", f"{(preds == y_test).float().mean().item():.4f}")

# (b) Confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10), CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10), CLASSES)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Fashion-MNIST confusion matrix")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()*0.5 else "black", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

# Ignore diagonal entries and find the two largest directed confusions.
offdiag = cm.copy()
np.fill_diagonal(offdiag, 0)
pairs = np.dstack(np.unravel_index(np.argsort(offdiag.ravel())[::-1], offdiag.shape))[0]
top_pairs = []
seen = set()
for i, j in pairs:
    if offdiag[i, j] == 0:
        break
    key = tuple(sorted((int(i), int(j))))
    if key not in seen:
        seen.add(key)
        total_pair = cm[i, j] + cm[j, i]
        top_pairs.append((int(i), int(j), int(cm[i, j]), int(cm[j, i]), int(total_pair)))
    if len(top_pairs) == 2:
        break

print("\nAnswer 1(b) — two most confused class pairs:")
for a, b, ab, ba, total in top_pairs:
    print(f"  {CLASSES[a]} ↔ {CLASSES[b]}: {CLASSES[a]}→{CLASSES[b]}={ab}, "
          f"{CLASSES[b]}→{CLASSES[a]}={ba}, combined={total}")

# (c) Three hypotheses are deliberately stated before TCAV/dissection results.
hypotheses = [
    ("Vertical-stripe concept should matter for Trousers",
     "trousers contain long vertical leg structures, so vertical structure should be useful to the classifier."),
    ("Dot concept should matter for Sandal",
     "sandal images contain separated small regions/holes and open texture-like details, so a sparse dot-like concept may affect its representation."),
    ("Horizontal-stripe concept should matter for T-shirt",
     "the broad horizontal torso/hem structure of T-shirts can produce horizontal edge responses.")
]
print("\nAnswer 1(c) — three hypotheses:")
for i, (h, why) in enumerate(hypotheses, 1):
    print(f"{i}. {h} because {why}")


In [ ]:
rng = np.random.default_rng(SEED)
H = W = 28

def _finish(img):
    img = ndimage.gaussian_filter(img, sigma=0.6)
    return np.clip(img / (img.max() + 1e-8), 0, 1)

def make_set(fn, n):
    imgs = np.stack([fn() for _ in range(n)])[:, None]          # (n,1,28,28) in [0,1]
    return normalize(torch.tensor(imgs, dtype=torch.float32))

def draw_line(img, r0, c0, r1, c1, thick):
    rr, cc = np.ogrid[:H, :W]
    for t in np.linspace(0, 1, 60):
        r, c = r0 + t * (r1 - r0), c0 + t * (c1 - c0)
        img[(rr - r) ** 2 + (cc - c) ** 2 <= (thick / 2) ** 2] = 1.0
    return img

def make_h_stripes():                                            # WORKED EXAMPLE
    img = np.zeros((H, W))
    gap, thick, off = rng.integers(4, 8), rng.integers(1, 3), rng.integers(0, 4)
    for r in range(off, H, gap):
        img[r:r + thick, 3:25] = 1.0
    return _finish(img)

def make_random():                                               # COUNTER-EXAMPLES
    img = np.zeros((H, W))
    for _ in range(rng.integers(1, 4)):
        r0, c0 = rng.uniform(7, 21, size=2)
        ang, L = rng.uniform(0, 2 * np.pi), rng.uniform(8, 16)
        r1 = np.clip(r0 + L * np.sin(ang), 2, 25); c1 = np.clip(c0 + L * np.cos(ang), 2, 25)
        draw_line(img, r0, c0, r1, c1, rng.uniform(2, 3.5))
    return _finish(img)

In [ ]:
# Part 2.2 — Concept generators

def make_v_stripes():
    """Randomised vertical stripe concept."""
    img = np.zeros((H, W))
    gap = int(rng.integers(4, 8))
    thick = int(rng.integers(1, 3))
    off = int(rng.integers(0, gap))
    tilt = rng.uniform(-2.5, 2.5)
    for c in range(-4 + off, W + 4, gap):
        draw_line(img, 3, c, 25, c + tilt, thick)
    return _finish(img)

def make_dots():
    """Randomised sparse-dot concept."""
    img = np.zeros((H, W))
    n = int(rng.integers(5, 13))
    radius = float(rng.uniform(1.0, 2.0))
    rr, cc = np.ogrid[:H, :W]
    for _ in range(n):
        r, c = rng.uniform(4, 24, size=2)
        img[(rr-r)**2 + (cc-c)**2 <= radius**2] = 1.0
    return _finish(img)

def make_custom():
    """Custom diagonal concept: a mixture of ±45° strokes."""
    img = np.zeros((H, W))
    n = int(rng.integers(2, 5))
    for _ in range(n):
        r0, c0 = rng.uniform(4, 20, size=2)
        length = rng.uniform(8, 17)
        sign = rng.choice([-1.0, 1.0])
        r1 = np.clip(r0 + sign * length * 0.75, 2, 25)
        c1 = np.clip(c0 + length * 0.75, 2, 25)
        draw_line(img, r0, c0, r1, c1, rng.uniform(1.5, 3.0))
    return _finish(img)

N_CONCEPT, N_RANDOM_SETS = 100, 10
concepts = {
    "h_stripes": make_set(make_h_stripes, N_CONCEPT),
    "v_stripes": make_set(make_v_stripes, N_CONCEPT),
    "dots": make_set(make_dots, N_CONCEPT),
    "diagonal": make_set(make_custom, N_CONCEPT),
}
random_sets = [make_set(make_random, N_CONCEPT) for _ in range(N_RANDOM_SETS)]

# Verify equal sizes and normalization convention.
assert all(v.shape == (N_CONCEPT, 1, 28, 28) for v in concepts.values())
assert all(v.shape == (N_CONCEPT, 1, 28, 28) for v in random_sets)
assert all(torch.isfinite(v).all() for v in list(concepts.values()) + random_sets)

print("Concept sets:", {k: tuple(v.shape) for k, v in concepts.items()})
print("Random sets:", len(random_sets), "sets of", tuple(random_sets[0].shape))
print("All sets use the same normalize(x) transformation.")

# Visualise 8 examples from every concept plus one random set.
nrows = len(concepts) + 1
fig, axes = plt.subplots(nrows, 8, figsize=(11, 1.9*nrows))
for row, (name, imgs) in enumerate(list(concepts.items()) + [("random", random_sets[0])]):
    for j, ax in enumerate(axes[row]):
        ax.imshow(imgs[j, 0] * STD + MEAN, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        if j == 0:
            ax.set_ylabel(name, rotation=0, labelpad=45, va="center")
plt.suptitle("Synthetic concept sets and random counter-examples", y=1.01)
plt.tight_layout(); plt.show()

print("\nAnswer 2.2(c):")
print("Every set contains the same number of images and uses the same normalization. "
      "Mismatched brightness, sparsity, thickness, or other low-level statistics could let "
      "the CAV separate concept from random images using a confound rather than the intended concept.")


In [ ]:
# Part 2.3 — Prediction histograms

fig, axes = plt.subplots(1, len(concepts), figsize=(15, 3.2), sharey=True)
for ax, (name, imgs) in zip(axes, concepts.items()):
    p = predict(model, imgs)
    counts = np.bincount(p.numpy(), minlength=10)
    ax.bar(np.arange(10), counts / len(p))
    ax.set_title(name)
    ax.set_xticks(range(10))
    ax.set_xticklabels([str(i) for i in range(10)], fontsize=8)
    ax.set_xlabel("Predicted class")
axes[0].set_ylabel("Fraction of concept images")
fig.suptitle("Model predictions on synthetic concept images")
plt.tight_layout(); plt.show()

concept_pred_summary = {}
for name, imgs in concepts.items():
    p = predict(model, imgs).numpy()
    order = np.argsort(np.bincount(p, minlength=10))[::-1]
    concept_pred_summary[name] = [(int(k), int((p == k).sum())) for k in order[:3]]

print("\nAnswer 2.3:")
for name, top in concept_pred_summary.items():
    print(f"  {name}: top predicted classes/counts = {top}")
print("These histograms can reveal what the classifier associates with the synthetic images, "
      "but they are not TCAV: a prediction histogram measures final class outputs, whereas "
      "TCAV measures directional sensitivity of a class score to a concept in an internal layer.")


In [ ]:
@torch.no_grad()
def get_acts(x, layer, batch=256):
    out = [model.features_upto(x[i:i+batch].to(device), layer).flatten(1).cpu()
           for i in range(0, len(x), batch)]
    return torch.cat(out).numpy()

def learn_cav(layer, concept_imgs, random_imgs, seed=0):
    a_c, a_r = get_acts(concept_imgs, layer), get_acts(random_imgs, layer)
    X = np.concatenate([a_c, a_r]); y = np.r_[np.ones(len(a_c)), np.zeros(len(a_r))]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = LogisticRegression(C=0.1, max_iter=2000).fit(Xtr, ytr)
    cav = clf.coef_[0] / np.linalg.norm(clf.coef_[0])
    return cav, clf.score(Xte, yte)

def class_grads(layer, k, n=300):
    """Gradient of logit_k w.r.t. the layer activations for n test images of class k."""
    x = normalize(X_test_raw[y_test == k][:n, None]).to(device)
    a = model.features_upto(x, layer).detach().requires_grad_(True)
    logit_k = model.forward_from(a, layer)[:, k]
    grad, = torch.autograd.grad(logit_k.sum(), a)
    return grad.flatten(1).cpu().numpy()

def tcav_score(grads, cav):
    return float(np.mean(grads @ cav > 0))

In [ ]:
# Part 3.2 — CAV quality

cav_acc = pd.DataFrame(index=list(concepts.keys()), columns=SmallCNN.LAYERS, dtype=float)
for name, imgs in concepts.items():
    for layer in SmallCNN.LAYERS:
        _, acc = learn_cav(layer, imgs, random_sets[0], seed=SEED)
        cav_acc.loc[name, layer] = acc

display(cav_acc.round(3))
print("\nAnswer 3.2:")
best_layer = cav_acc.mean(axis=0).idxmax()
print(f"The best mean held-out CAV separation in this run is at {best_layer} "
      f"(mean accuracy = {cav_acc[best_layer].mean():.3f}).")
print("A high CAV accuracy only means the concept images are linearly separable from the "
      "random counter-examples in that representation. It does not show that the classifier "
      "uses that concept for any particular class; TCAV additionally measures the sign of the "
      "class-logit directional derivative.")


In [ ]:
# Part 3.3 — Full TCAV with null significance and Bonferroni correction

def tcav_experiment(layer, classes=range(10)):
    """
    For each concept/class pair:
      - 10 CAVs: concept vs each random set
      - 10 null CAVs: random set i vs a different random set
      - report mean, std, raw p and Bonferroni significance
    """
    cavs = {
        c: [learn_cav(layer, imgs, r, seed=i)[0]
            for i, r in enumerate(random_sets)]
        for c, imgs in concepts.items()
    }

    # Use non-overlapping random-set pairs for a clean null where possible.
    null_cavs = [
        learn_cav(layer, random_sets[i], random_sets[(i + 1) % N_RANDOM_SETS], seed=100+i)[0]
        for i in range(N_RANDOM_SETS)
    ]

    results = {}
    for k in classes:
        g = class_grads(layer, k)
        null_scores = np.array([tcav_score(g, v) for v in null_cavs])
        for c in concepts:
            scores = np.array([tcav_score(g, v) for v in cavs[c]])
            test = stats.ttest_ind(scores, null_scores, equal_var=False)
            results[(c, k)] = {
                "mean": float(scores.mean()),
                "std": float(scores.std(ddof=1)),
                "p": float(test.pvalue),
                "scores": scores,
                "null_scores": null_scores,
            }
    return results

def add_bonferroni(results, alpha=0.05, n_tests_total=None):
    n_tests = len(results) if n_tests_total is None else n_tests_total
    alpha_bonf = alpha / n_tests
    for r in results.values():
        r["alpha_bonf"] = alpha_bonf
        r["significant"] = r["p"] < alpha_bonf
    return results, alpha_bonf

def plot_tcav(results, layer):
    alpha_bonf = next(iter(results.values()))["alpha_bonf"]
    names = list(concepts)
    fig, ax = plt.subplots(figsize=(14, 4.2))
    w = 0.8 / len(names)
    for j, c in enumerate(names):
        means = [results[(c,k)]["mean"] for k in range(10)]
        stds  = [results[(c,k)]["std"] for k in range(10)]
        sigs  = [results[(c,k)]["significant"] for k in range(10)]
        for k in range(10):
            bar = ax.bar(k + (j-(len(names)-1)/2)*w, means[k], w*0.9,
                         yerr=stds[k], capsize=2,
                         label=c if k == 0 else None)
            if not sigs[k]:
                bar[0].set_hatch("//")
                bar[0].set_alpha(0.65)
    ax.axhline(0.5, color=C_GRAY, lw=1, ls="--")
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=35, ha="right")
    ax.set_xlabel("Target class"); ax.set_ylabel("TCAV score")
    ax.set_ylim(0, 1.08)
    ax.set_title(f"TCAV at {layer} — hatched = not Bonferroni-significant (α={alpha_bonf:.2e})")
    ax.legend(ncol=4, frameon=False, bbox_to_anchor=(0,1.18), loc="upper left")
    plt.tight_layout(); plt.show()

tcav_results = {layer: tcav_experiment(layer) for layer in SmallCNN.LAYERS}

# Correct across all concept × class × layer tests in this experiment.
N_TCAV_TESTS = sum(len(r) for r in tcav_results.values())
GLOBAL_ALPHA_BONF = 0.05 / N_TCAV_TESTS
for layer, res in tcav_results.items():
    for r in res.values():
        r["alpha_bonf"] = GLOBAL_ALPHA_BONF
        r["significant"] = r["p"] < GLOBAL_ALPHA_BONF

for layer, res in tcav_results.items():
    plot_tcav(res, layer)
    sig_raw = sum(r["p"] < 0.05 for r in res.values())
    sig_bonf = sum(r["significant"] for r in res.values())
    print(f"{layer}: {sig_raw} raw-significant results; {sig_bonf} survive global Bonferroni; "
          f"{sig_raw - sig_bonf} raw results are lost after correction.")

print(f"\nGlobal Bonferroni correction: alpha = 0.05 / {N_TCAV_TESTS} = {GLOBAL_ALPHA_BONF:.3g}.")


In [ ]:
N_DISSECT = 1000
Xd = X_test_raw[:N_DISSECT].numpy()
yd = y_test[:N_DISSECT].numpy()

def concept_masks(img, label):
    fg = img > 0.3
    gy, gx = ndimage.sobel(img, axis=0), ndimage.sobel(img, axis=1)
    # Filled holes: regions inside foreground boundaries.
    hole = ndimage.binary_fill_holes(fg) & ~fg

    # Upper half of the foreground: useful as a simple part-level concept.
    upper = np.zeros_like(fg)
    upper[:14] = fg[:14]

    # A diagonal-gradient/edge concept.
    diag = (np.abs(gx + gy) > 0.8) | (np.abs(gx - gy) > 0.8)

    m = {
        "foreground": fg,
        "background": ~fg,
        "h-edge": (np.abs(gy) > 0.7) & (np.abs(gy) > np.abs(gx)),
        "v-edge": (np.abs(gx) > 0.7) & (np.abs(gx) > np.abs(gy)),
        "hole": hole,
        "upper-half": upper,
        "diagonal": diag & fg,
    }
    for d in range(10):
        m[f"class-{d}"] = fg if label == d else np.zeros_like(fg)
    return m

CONCEPTS = list(concept_masks(Xd[0], yd[0]).keys())
L = torch.tensor(np.stack([
    np.stack([concept_masks(im, lb)[c] for c in CONCEPTS])
    for im, lb in zip(Xd, yd)
]))

print("Concepts:", CONCEPTS, "| mask tensor:", tuple(L.shape))

# Pixel fraction across the dissection set.
fractions = L.float().mean(dim=(0,2,3)).numpy()
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(CONCEPTS, fractions)
ax.set_ylabel("Mean pixel fraction")
ax.set_xlabel("Concept mask")
ax.set_title("Average pixel fraction of dissection concepts")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()

# Visualise masks for three classes.
show = [np.where(yd == d)[0][0] for d in (0, 1, 5)]
show_names = [CLASSES[d] for d in (0,1,5)]
mask_names = ["foreground", "background", "h-edge", "v-edge", "hole", "upper-half", "diagonal"]
fig, axes = plt.subplots(len(show), len(mask_names)+1, figsize=(14, 6))
for r, i in enumerate(show):
    axes[r,0].imshow(Xd[i], cmap="gray", vmin=0, vmax=1)
    axes[r,0].set_title(show_names[r])
    for j, name in enumerate(mask_names, 1):
        axes[r,j].imshow(L[i, CONCEPTS.index(name)].float(), cmap="Blues")
        if r == 0: axes[r,j].set_title(name, fontsize=8)
    for ax in axes[r]:
        ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
@torch.no_grad()
def dissect(net, layer, quantile=0.995, iou_thresh=0.04, batch=200):
    net.eval()
    x = normalize(torch.tensor(Xd)[:, None]).to(device)
    A = torch.cat([net.features_upto(x[i:i+batch], layer).cpu() for i in range(0, len(x), batch)])
    n_units = A.shape[1]
    flat = A.transpose(0, 1).reshape(n_units, -1).numpy()
    T = torch.tensor(np.quantile(flat, quantile, axis=1), dtype=torch.float32)

    inter = torch.zeros(n_units, len(CONCEPTS)); size_M = torch.zeros(n_units)
    size_L = L.flatten(2).sum((0, 2)).float()
    for i in range(0, len(A), batch):
        up = F.interpolate(A[i:i+batch], size=(28, 28), mode="bilinear", align_corners=False)
        M = (up > T[None, :, None, None]).flatten(2).float()
        Lb = L[i:i+batch].flatten(2).float()
        inter += torch.einsum("bup,bcp->uc", M, Lb)
        size_M += M.sum((0, 2))
    iou = inter / (size_M[:, None] + size_L[None, :] - inter + 1e-8)
    best = iou.argmax(1)
    rows = [{"unit": u, "concept": CONCEPTS[best[u]], "iou": iou[u, best[u]].item(),
             "detector": iou[u, best[u]].item() > iou_thresh} for u in range(n_units)]
    return rows, A, T, iou

dissection = {layer: dissect(model, layer) for layer in SmallCNN.LAYERS}
for layer, (rows, _, _, _) in dissection.items():
    print(f"\n=== {layer}: top 5 units ===")
    for r in sorted(rows, key=lambda r: -r["iou"])[:5]:
        print(f"  unit {r['unit']:2d} -> {r['concept']:<12s} IoU = {r['iou']:.3f}")

In [ ]:
# Part 4.3 — Inspect units

def show_unit(layer, unit, n=8):
    rows, A, T, _ = dissection[layer]
    top = A[:, unit].flatten(1).max(1).values.argsort(descending=True)[:n].tolist()
    fig, axes = plt.subplots(1, n, figsize=(1.55*n, 2.1))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, top):
        up = F.interpolate(A[i:i+1, unit:unit+1], size=(28,28),
                           mode="bilinear", align_corners=False)[0,0]
        ax.imshow(Xd[i], cmap="gray", vmin=0, vmax=1)
        ax.contour((up > T[unit]).numpy(), levels=[0.5],
                   colors=C_ORANGE, linewidths=1.5)
        ax.set_title(f"{CLASSES[yd[i]]}\n#{i}", fontsize=8)
        ax.axis("off")
    r = rows[unit]
    fig.suptitle(f"{layer} unit {unit}: {r['concept']} (IoU={r['iou']:.3f})", y=1.05)
    plt.tight_layout(); plt.show()
    return r

selected_units = {}
for layer, (rows, _, _, _) in dissection.items():
    best = max(rows, key=lambda r: r["iou"])
    selected_units[layer] = best["unit"]
    print(f"Best {layer} unit: {best['unit']} → {best['concept']} (IoU={best['iou']:.3f})")
    show_unit(layer, best["unit"])

best_layer = max(SmallCNN.LAYERS,
                 key=lambda l: max(r["iou"] for r in dissection[l][0]))
best_unit = selected_units[best_layer]
best_row = dissection[best_layer][0][best_unit]

print("\nAnswer 4.3(b):")
print(f"The strongest labelled unit in this run is {best_layer} unit {best_unit}, "
      f"labelled '{best_row['concept']}' with IoU={best_row['iou']:.3f}. "
      "The label is convincing only if the outlined regions repeatedly fall on the "
      "same visual structure across the eight examples; otherwise the IoU label may "
      "be a proxy for a correlated cue rather than the intended concept.")


In [ ]:
# Part 4.4 — Concepts across depth, untrained control, threshold sensitivity

CATEGORY = {
    "foreground": "colour",
    "background": "colour",
    "h-edge": "edge",
    "v-edge": "edge",
    "hole": "part",
    "upper-half": "part",
    "diagonal": "part",
}
CATEGORY.update({f"class-{d}": "object" for d in range(10)})
CATS = ["colour", "edge", "part", "object"]

def category_counts(dis):
    return {
        layer: [
            sum(r["detector"] and CATEGORY.get(r["concept"]) == c for r in rows)
            for c in CATS
        ]
        for layer, (rows, _, _, _) in dis.items()
    }

def plot_counts(dis, title, ax=None):
    counts = category_counts(dis)
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 3.5))
    w = 0.18
    for j, c in enumerate(CATS):
        ax.bar(np.arange(3) + (j-1.5)*w,
               [counts[l][j] for l in SmallCNN.LAYERS],
               w*0.9, label=c)
    ax.set_xticks(range(3), SmallCNN.LAYERS)
    ax.set_ylabel("# detector units (IoU > 0.04)")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
    return counts

random_model = SmallCNN().to(device).eval()
dissection_rand = {layer: dissect(random_model, layer) for layer in SmallCNN.LAYERS}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
trained_counts = plot_counts(dissection, "Trained network", axes[0])
random_counts = plot_counts(dissection_rand, "Untrained random network", axes[1])
plt.tight_layout(); plt.show()

print("Trained category counts:", trained_counts)
print("Untrained category counts:", random_counts)

# Threshold sensitivity on conv2.
base_rows = dissection["conv2"][0]
base_labels = [r["concept"] for r in base_rows]
sens = {}
for q in [0.99, 0.999]:
    q_rows, _, _, _ = dissect(model, "conv2", quantile=q)
    q_labels = [r["concept"] for r in q_rows]
    same = np.mean(np.array(q_labels) == np.array(base_labels)) * 100
    sens[q] = same
    print(f"conv2 threshold q={q}: {same:.1f}% of unit labels unchanged vs q=0.995")

print("\nAnswer 4.4:")
print("Interpret hierarchy from the detector-count plots rather than assuming it. "
      "A trained network showing more edge/part/object structure at deeper layers than "
      "the random control is consistent with learned representations becoming more "
      "semantically organized. Random networks can still have non-zero IoUs because "
      "random filters produce structured activation patterns and the masks themselves "
      "are not independent of image structure.")


In [ ]:
# Part 5.1 — Unit-direction TCAV at conv3

# Prefer the top three units whose best dissection label is class-k/object.
class_units = [
    r for r in dissection["conv3"][0]
    if r["concept"].startswith("class-")
]
if len(class_units) >= 3:
    chosen_rows = sorted(class_units, key=lambda r: -r["iou"])[:3]
else:
    chosen_rows = sorted(dissection["conv3"][0], key=lambda r: -r["iou"])[:3]

chosen_units = [r["unit"] for r in chosen_rows]
D = 64 * 7 * 7

def unit_cav(unit):
    v = np.zeros(D, dtype=np.float32)
    v[unit*49:(unit+1)*49] = 1.0
    return v / np.linalg.norm(v)

unit_tcav = {}
rng_local = np.random.default_rng(SEED + 123)
all_units = np.arange(64)

for unit in chosen_units:
    cav = unit_cav(unit)
    scores = np.array([
        tcav_score(class_grads("conv3", k), cav) for k in range(10)
    ])
    null_units = rng_local.choice(np.setdiff1d(all_units, [unit]), size=10, replace=False)
    null_scores = np.array([
        tcav_score(class_grads("conv3", k), unit_cav(nu))
        for k in range(10) for nu in null_units
    ]).reshape(10, 10)

    # The observed unit gives one score per class; compare it with the null
    # distribution using a one-sided empirical tail probability.
    pvals = np.array([
        (1 + np.sum(null_scores[:, k] >= scores[k])) / (len(null_scores) + 1)
        for k in range(10)
    ])
    unit_tcav[unit] = {"scores": scores, "pvals": pvals, "null": null_scores}

    top_k = int(scores.argmax())
    print(f"unit {unit} | dissection={dissection['conv3'][0][unit]['concept']} "
          f"| top TCAV class={CLASSES[top_k]} score={scores[top_k]:.3f} "
          f"| p={pvals[top_k]:.3g}")

# Visualise scores for selected units.
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(10)
w = 0.24
for j, unit in enumerate(chosen_units):
    ax.bar(x + (j-1)*w, unit_tcav[unit]["scores"], w, label=f"unit {unit}")
ax.axhline(0.5, color=C_GRAY, ls="--", lw=1)
ax.set_xticks(x, CLASSES, rotation=35, ha="right")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Target class")
ax.set_ylabel("Unit-direction TCAV score")
ax.set_title("Unit-direction TCAV for selected conv3 units")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()


In [ ]:
# Part 5.2 — Ablation

@torch.no_grad()
def accuracy_with_ablation(layer, unit):
    """Per-class accuracy after zeroing one channel at the chosen layer."""
    model.eval()
    correct = np.zeros(10, dtype=np.int64)
    total = np.zeros(10, dtype=np.int64)

    for i in range(0, len(X_test_raw), 512):
        raw = X_test_raw[i:i+512]
        y = y_test[i:i+512].numpy()
        x = normalize(raw[:, None]).to(device)

        a = model.features_upto(x, layer).clone()
        a[:, unit, :, :] = 0.0
        pred = model.forward_from(a, layer).argmax(1).cpu().numpy()

        for k in range(10):
            mask = (y == k)
            total[k] += mask.sum()
            correct[k] += (pred[mask] == k).sum()

    return correct / np.maximum(total, 1)

baseline_per_class = per_class_acc.copy()

# Ablate selected units.
ablation_drops = {}
for unit in chosen_units:
    ab = accuracy_with_ablation("conv3", unit)
    ablation_drops[unit] = baseline_per_class - ab

# Ten random-unit controls.
control_units = rng_local.choice(np.setdiff1d(np.arange(64), chosen_units), size=10, replace=False)
control_drops = np.vstack([
    baseline_per_class - accuracy_with_ablation("conv3", int(u))
    for u in control_units
])

fig, ax = plt.subplots(figsize=(13, 4.5))
w = 0.75 / max(len(chosen_units), 1)
for j, unit in enumerate(chosen_units):
    ax.bar(np.arange(10) + (j-(len(chosen_units)-1)/2)*w,
           ablation_drops[unit], w*0.9, label=f"selected unit {unit}")
ax.errorbar(np.arange(10) + 0.36, control_drops.mean(0),
            yerr=control_drops.std(0, ddof=1), fmt="o", capsize=3,
            label="10 random units: mean ± SD")
ax.axhline(0, color=C_GRAY, lw=1)
ax.set_xticks(range(10), CLASSES, rotation=35, ha="right")
ax.set_ylabel("Accuracy drop (baseline − ablated)")
ax.set_xlabel("Class")
ax.set_title("Conv3 unit ablation vs random-unit control")
ax.legend(frameon=False, ncol=2)
plt.tight_layout(); plt.show()

print("Selected units:", chosen_units)
print("Random control units:", control_units.tolist())
